<a href="https://colab.research.google.com/github/rajesh-doma/finbot/blob/main/notebooks/02_chunking_qdrant%2Cipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — Chunking & Qdrant Ingestion
Chunks all documents from all 5 collections and stores embeddings in Qdrant Cloud with full RBAC metadata.

Install python packages need for chucking and ingestion

In [1]:
!pip install -q qdrant-client sentence-transformers docling pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 103.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 93.3 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 232.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 111.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 446.3/446.3 kB 45.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 76.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 86.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 598.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 27.5 MB/s eta 0:00:00
  

Load the API secrets needs to access the groq LLM and Vector DB Qdrant URL and API Key

In [2]:
#Load Secrets
from google.colab import userdata

QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
GROQ_API_KEY   = userdata.get("GROQ_API_KEY")

print(f"QDRANT_URL     : {QDRANT_URL}")
print(f"QDRANT_API_KEY : {'*' * len(QDRANT_API_KEY)}")
print(f"GROQ_API_KEY   : {'*' * len(GROQ_API_KEY)}")

QDRANT_URL     : https://319ca0b0-7d9a-48b4-8b9c-2148c9e71bfe.sa-east-1-0.aws.cloud.qdrant.io
QDRANT_API_KEY : ****************************************************************************************************
GROQ_API_KEY   : ********************************************************


Clone Data from GitHub

In [3]:
import os

if not os.path.exists('/content/finbot'):
    !git clone --no-checkout --depth 1 https://github.com/rajesh-doma/finbot.git /content/finbot
    %cd /content/finbot
    !git sparse-checkout init --cone
    !git sparse-checkout set data
    !git checkout main
else:
    %cd /content/finbot
    !git pull

print("Data ready ✅")
!find /content/finbot/data -type f | grep -v .gitkeep | sort

Cloning into '/content/finbot'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 36 (delta 4), reused 30 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 1.09 MiB | 4.71 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/finbot
Already on 'main'
Your branch is up to date with 'origin/main'.
Data ready ✅
/content/finbot/data/engineering/engineering_master_doc.md
/content/finbot/data/engineering/incident_report_log.md
/content/finbot/data/engineering/sprint_metrics_2024.md
/content/finbot/data/engineering/system_sla_report_2024.md
/content/finbot/data/finance/department_budget_2024.docx
/content/finbot/data/finance/financial_summary.docx
/content/finbot/data/finance/quarterly_financial_report.docx
/content/finbot/data/finance/vendor_payments_summary.docx
/content/finbot/data/general/employee_handbook.pdf
/content/finbot/data/hr/hr_data.csv
/content/finbot/data/mar

Security(RBAC) Config

In [4]:
COLLECTION_ROLES = {
    "general":     ["employee", "finance", "engineering", "marketing", "hr", "c_level"],
    "finance":     ["finance", "c_level"],
    "engineering": ["engineering", "c_level"],
    "marketing":   ["marketing", "c_level"],
    "hr":          ["hr", "c_level"],
}

# All files mapped to their collection
ALL_FILES = [
    # general
    ("/content/finbot/data/general/employee_handbook.pdf", "general"),

    # finance
    ("/content/finbot/data/finance/department_budget_2024.docx",    "finance"),
    ("/content/finbot/data/finance/financial_summary.docx",         "finance"),
    ("/content/finbot/data/finance/quarterly_financial_report.docx","finance"),
    ("/content/finbot/data/finance/vendor_payments_summary.docx",   "finance"),

    # engineering
    ("/content/finbot/data/engineering/engineering_master_doc.md",  "engineering"),
    ("/content/finbot/data/engineering/incident_report_log.md",     "engineering"),
    ("/content/finbot/data/engineering/sprint_metrics_2024.md",     "engineering"),
    ("/content/finbot/data/engineering/system_sla_report_2024.md",  "engineering"),

    # hr
    ("/content/finbot/data/hr/hr_data.csv", "hr"),

    # marketing
    ("/content/finbot/data/marketing/campaign_performance_data.docx",  "marketing"),
    ("/content/finbot/data/marketing/customer_acquisition_report.docx","marketing"),
    ("/content/finbot/data/marketing/marketing_report_2024.docx",      "marketing"),
    ("/content/finbot/data/marketing/marketing_report_q1_2024.docx",   "marketing"),
    ("/content/finbot/data/marketing/marketing_report_q2_2024.docx",   "marketing"),
    ("/content/finbot/data/marketing/marketing_report_q3_2024.docx",   "marketing"),
    ("/content/finbot/data/marketing/marketing_report_q4_2024.docx",   "marketing"),
]

print(f"Total files to ingest: {len(ALL_FILES)}")
for f, c in ALL_FILES:
    print(f"  {c:15s} → {os.path.basename(f)}")

Total files to ingest: 17
  general         → employee_handbook.pdf
  finance         → department_budget_2024.docx
  finance         → financial_summary.docx
  finance         → quarterly_financial_report.docx
  finance         → vendor_payments_summary.docx
  engineering     → engineering_master_doc.md
  engineering     → incident_report_log.md
  engineering     → sprint_metrics_2024.md
  engineering     → system_sla_report_2024.md
  hr              → hr_data.csv
  marketing       → campaign_performance_data.docx
  marketing       → customer_acquisition_report.docx
  marketing       → marketing_report_2024.docx
  marketing       → marketing_report_q1_2024.docx
  marketing       → marketing_report_q2_2024.docx
  marketing       → marketing_report_q3_2024.docx
  marketing       → marketing_report_q4_2024.docx


Define Parser and Chunker Functions

In [5]:
import hashlib
import pandas as pd
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

# Docling converter
pipeline_opts = PdfPipelineOptions()
pipeline_opts.do_ocr = False
pipeline_opts.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts)
    }
)

def _get_page(item):
    try:
        return item.prov[0].page_no if item.prov else 0
    except:
        return 0

def parse_document_flat(doc, source_file, collection):
    """Parse Docling document into flat chunks with RBAC metadata."""
    access_roles = COLLECTION_ROLES.get(collection, [])
    chunks = []

    all_levels = [
        level for item, level in doc.iterate_items()
        if type(item).__name__ == "SectionHeaderItem"
    ]
    min_level = min(all_levels) if all_levels else 1

    current_section = os.path.basename(source_file)
    current_page    = 0
    buffer          = []

    def flush(section, page, buf, chunk_type="text"):
        if not buf:
            return
        content = "\n".join(buf).strip()
        if not content:
            return
        chunk_id = hashlib.md5(
            f"{source_file}::{section}::{content[:50]}".encode()
        ).hexdigest()
        chunks.append({
            "chunk_id":        chunk_id,
            "text":            f"[{section}]\n{content}",
            "source_document": os.path.basename(source_file),
            "collection":      collection,
            "access_roles":    access_roles,
            "section_title":   section,
            "page_number":     page,
            "chunk_type":      chunk_type,
            "parent_chunk_id": None,
        })
        buf.clear()

    for item, level in doc.iterate_items():
        item_type = type(item).__name__

        if item_type == "SectionHeaderItem":
            flush(current_section, current_page, buffer)
            current_section = item.text.strip()
            current_page    = _get_page(item)

        elif item_type == "TextItem":
            text = item.text.strip()
            if text:
                if sum(len(t) for t in buffer) > 1200:
                    flush(current_section, current_page, buffer)
                buffer.append(text)
                current_page = _get_page(item) or current_page

        elif item_type == "TableItem":
            flush(current_section, current_page, buffer)
            try:
                table_md = item.export_to_markdown()
            except:
                table_md = str(item)
            chunk_id = hashlib.md5(
                f"{source_file}::table::{current_section}".encode()
            ).hexdigest()
            chunks.append({
                "chunk_id":        chunk_id,
                "text":            f"[{current_section}]\n{table_md}",
                "source_document": os.path.basename(source_file),
                "collection":      collection,
                "access_roles":    access_roles,
                "section_title":   current_section,
                "page_number":     current_page,
                "chunk_type":      "table",
                "parent_chunk_id": None,
            })

    flush(current_section, current_page, buffer)
    return chunks


def parse_csv(filepath, collection):
    """Parse CSV into chunks — one chunk per 20 rows."""
    access_roles = COLLECTION_ROLES.get(collection, [])
    chunks = []
    df = pd.read_csv(filepath, dtype=str).fillna("")
    source_file = os.path.basename(filepath)
    chunk_size  = 20

    for i in range(0, len(df), chunk_size):
        chunk_df  = df.iloc[i:i + chunk_size]
        table_md  = chunk_df.to_markdown(index=False)
        section   = f"Rows {i+1} to {i+len(chunk_df)}"
        chunk_id  = hashlib.md5(f"{source_file}::{section}".encode()).hexdigest()
        chunks.append({
            "chunk_id":        chunk_id,
            "text":            f"[{source_file} - {section}]\n{table_md}",
            "source_document": source_file,
            "collection":      collection,
            "access_roles":    access_roles,
            "section_title":   section,
            "page_number":     0,
            "chunk_type":      "table",
            "parent_chunk_id": None,
        })

    return chunks


def parse_file(filepath, collection):
    """Route file to correct parser based on extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext in (".csv", ".tsv"):
        return parse_csv(filepath, collection)
    else:
        result = converter.convert(filepath)
        return parse_document_flat(result.document, filepath, collection)


print("Parser and chunker ready ✅")

Parser and chunker ready ✅


Parse All Input Files and create chunks

In [6]:
from tqdm.notebook import tqdm

all_chunks = []
parse_stats = {}

for filepath, collection in tqdm(ALL_FILES, desc="Parsing files"):
    filename = os.path.basename(filepath)
    try:
        chunks = parse_file(filepath, collection)
        all_chunks.extend(chunks)
        parse_stats[filename] = f"✅ {len(chunks)} chunks"
    except Exception as e:
        parse_stats[filename] = f"❌ ERROR: {e}"

print(f"\n{'='*50}")
print(f"Total chunks : {len(all_chunks)}")
print(f"\nPer file:")
for fname, status in parse_stats.items():
    print(f"  {fname:45s} → {status}")

Parsing files:   0%|          | 0/17 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Total chunks : 568

Per file:
  employee_handbook.pdf                         → ✅ 54 chunks
  department_budget_2024.docx                   → ✅ 38 chunks
  financial_summary.docx                        → ✅ 27 chunks
  quarterly_financial_report.docx               → ✅ 55 chunks
  vendor_payments_summary.docx                  → ✅ 41 chunks
  engineering_master_doc.md                     → ✅ 61 chunks
  incident_report_log.md                        → ✅ 23 chunks
  sprint_metrics_2024.md                        → ✅ 38 chunks
  system_sla_report_2024.md                     → ✅ 46 chunks
  hr_data.csv                                   → ✅ 25 chunks
  campaign_performance_data.docx                → ✅ 20 chunks
  customer_acquisition_report.docx              → ✅ 37 chunks
  marketing_report_2024.docx                    → ✅ 28 chunks
  marketing_report_q1_2024.docx                 → ✅ 13 chunks
  marketing_report_q2_2024.docx                 → ✅ 19 chunks
  marketing_report_q3_2024.docx        

Load Embedding Model

In [7]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Model loaded ✅")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

Loading embedding model...
Model loaded ✅
Embedding dimension: 384


Initialise Qdrant Collection

In [8]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm

# Connect to Qdrant Cloud
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION = "finbot"
DIM        = embed_model.get_sentence_embedding_dimension()  # 384

# Drop and recreate collection (clean start)
existing = [c.name for c in client.get_collections().collections]
if COLLECTION in existing:
    client.delete_collection(COLLECTION)
    print(f"Deleted existing '{COLLECTION}' collection")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=qm.VectorParams(
        size=DIM,
        distance=qm.Distance.COSINE,
    ),
)

# Payload indexes for fast RBAC filtering
for field, schema in [
    ("access_roles",    qm.PayloadSchemaType.KEYWORD),
    ("collection",      qm.PayloadSchemaType.KEYWORD),
    ("chunk_type",      qm.PayloadSchemaType.KEYWORD),
    ("source_document", qm.PayloadSchemaType.KEYWORD),
]:
    client.create_payload_index(COLLECTION, field, schema)

print(f"Collection '{COLLECTION}' created ✅")
print(f"Dimension  : {DIM}")
print(f"Indexes    : access_roles, collection, chunk_type, source_document")

Deleted existing 'finbot' collection
Collection 'finbot' created ✅
Dimension  : 384
Indexes    : access_roles, collection, chunk_type, source_document


Embed and Upsert All Chunks

In [9]:
from tqdm.notebook import tqdm
import hashlib

BATCH_SIZE = 64

print(f"Embedding and upserting {len(all_chunks)} chunks...")

for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Upserting batches"):
    batch   = all_chunks[i:i + BATCH_SIZE]
    texts   = [c["text"] for c in batch]
    vectors = embed_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()

    points = []
    for chunk, vector in zip(batch, vectors):
        uid = int(hashlib.md5(chunk["chunk_id"].encode()).hexdigest(), 16) % (2**63)
        points.append(qm.PointStruct(
            id      = uid,
            vector  = vector,
            payload = {
                "source_document": chunk["source_document"],
                "collection":      chunk["collection"],
                "access_roles":    chunk["access_roles"],
                "section_title":   chunk["section_title"],
                "page_number":     chunk["page_number"],
                "chunk_type":      chunk["chunk_type"],
                "parent_chunk_id": chunk["parent_chunk_id"],
                "text":            chunk["text"],
            }
        ))

    client.upsert(
        collection_name=COLLECTION,
        points=points,
        wait=True,
    )

total = client.count(COLLECTION).count
print(f"\nUpsert complete ✅")
print(f"Total points in Qdrant: {total}")

Embedding and upserting 568 chunks...


Upserting batches:   0%|          | 0/9 [00:00<?, ?it/s]


Upsert complete ✅
Total points in Qdrant: 537


In [11]:
from qdrant_client.models import Filter, FieldCondition, MatchAny

query = "What are the Q3 revenue figures and budget allocations?"
q_vec = embed_model.encode([query], normalize_embeddings=True)[0].tolist()

print(f"Query: {query}\n")
print("="*60)

for role, allowed_collections in COLLECTION_ROLES.items():
    results = client.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="access_roles",
                    match=MatchAny(any=[role])
                )
            ]
        ),
        limit=3,
        with_payload=True,
    ).points

    print(f"Role [{role:12s}] → {len(results)} results")
    for r in results:
        print(f"  collection={r.payload['collection']:12s} "
              f"score={r.score:.3f}  "
              f"source={r.payload['source_document']}")
    print()

Query: What are the Q3 revenue figures and budget allocations?

Role [general     ] → 0 results

Role [finance     ] → 3 results
  collection=finance      score=0.554  source=quarterly_financial_report.docx
  collection=finance      score=0.514  source=quarterly_financial_report.docx
  collection=finance      score=0.505  source=financial_summary.docx

Role [engineering ] → 3 results
  collection=engineering  score=0.544  source=sprint_metrics_2024.md
  collection=engineering  score=0.492  source=sprint_metrics_2024.md
  collection=engineering  score=0.486  source=sprint_metrics_2024.md

Role [marketing   ] → 3 results
  collection=marketing    score=0.710  source=marketing_report_q3_2024.docx
  collection=marketing    score=0.622  source=marketing_report_q2_2024.docx
  collection=marketing    score=0.621  source=marketing_report_q1_2024.docx

Role [hr          ] → 3 results
  collection=general      score=0.385  source=employee_handbook.pdf
  collection=general      score=0.377  sourc

In [13]:
print("ADVERSARIAL RBAC TEST")
print("="*60)
print("Engineering user asking a finance question...\n")

finance_query = "What are the Q3 revenue figures and budget allocations?"
print(finance_query)
q_vec = embed_model.encode([finance_query], normalize_embeddings=True)[0].tolist()

# Step 1 — What finance role sees (should see finance docs)
finance_results = client.query_points(
    collection_name=COLLECTION,
    query=q_vec,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="access_roles",
                match=MatchAny(any=["finance"])
            )
        ]
    ),
    limit=3,
    with_payload=True,
).points

print(f"Finance role sees:")
for r in finance_results:
    print(f"  collection={r.payload['collection']:12s} "
          f"score={r.score:.3f}  "
          f"source={r.payload['source_document']}")

# Step 2 — What engineering role sees for SAME query (must NOT see finance docs)
print(f"\nEngineering role sees (same query):")
eng_results = client.query_points(
    collection_name=COLLECTION,
    query=q_vec,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="access_roles",
                match=MatchAny(any=["engineering"])
            )
        ]
    ),
    limit=3,
    with_payload=True,
).points

finance_leaked = [r for r in eng_results
                  if r.payload['collection'] == 'finance']

for r in eng_results:
    print(f"  collection={r.payload['collection']:12s} "
          f"score={r.score:.3f}  "
          f"source={r.payload['source_document']}")

# Step 3 — Verdict
print(f"\nVERDICT:")
if finance_leaked:
    print(f"  ❌ RBAC FAILED — {len(finance_leaked)} finance chunks leaked to engineering role")
else:
    print(f"  ✅ RBAC PASSED — engineering role cannot see any finance documents")
    print(f"  ✅ Finance data is invisible to engineering role at Qdrant query level")

ADVERSARIAL RBAC TEST
Engineering user asking a finance question...

What are the Q3 revenue figures and budget allocations?
Finance role sees:
  collection=finance      score=0.554  source=quarterly_financial_report.docx
  collection=finance      score=0.514  source=quarterly_financial_report.docx
  collection=finance      score=0.505  source=financial_summary.docx

Engineering role sees (same query):
  collection=engineering  score=0.544  source=sprint_metrics_2024.md
  collection=engineering  score=0.492  source=sprint_metrics_2024.md
  collection=engineering  score=0.486  source=sprint_metrics_2024.md

VERDICT:
  ✅ RBAC PASSED — engineering role cannot see any finance documents
  ✅ Finance data is invisible to engineering role at Qdrant query level


In [18]:
print("ADVERSARIAL TEST — Engineering user asking a Finance question")
print("="*60)

test_cases = [
    "What are the Q3 revenue figures and budget allocations?",
    "What is the total company budget for 2024?",
    "Show me the vendor payments and financial summary",
]

for query in test_cases:
    q_vec = embed_model.encode([query], normalize_embeddings=True)[0].tolist()

    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    for role in ["finance", "engineering"]:
        results = client.query_points(
            collection_name=COLLECTION,
            query=q_vec,
            query_filter=Filter(
                must=[
                    FieldCondition(
                        key="access_roles",
                        match=MatchAny(any=[role])
                    )
                ]
            ),
            limit=3,
            with_payload=True,
        ).points

        print(f"\n  ROLE: {role.upper()}")
        if not results:
            print("    → No results returned")
            continue

        for i, r in enumerate(results, 1):
            print(f"\n    Chunk {i}:")
            print(f"      collection   : {r.payload['collection']}")
            print(f"      source       : {r.payload['source_document']}")
            print(f"      section      : {r.payload['section_title']}")
            print(f"      page         : {r.payload['page_number']}")
            print(f"      score        : {r.score:.4f}")
            print(f"      access_roles : {r.payload['access_roles']}")
            print(f"      words        : {len(r.payload['text'].split())}")
            print(f"      text preview : {r.payload['text'][:2000]}")

        # Leakage check
        finance_leaked = [r for r in results
                         if r.payload['collection'] == 'finance']

        if role == "engineering":
            print(f"\n  VERDICT:")
            if finance_leaked:
                print(f"    ❌ RBAC FAILED — finance data leaked to engineering role")
            else:
                print(f"    ✅ RBAC PASSED — engineering role cannot see finance documents")

ADVERSARIAL TEST — Engineering user asking a Finance question

Query: What are the Q3 revenue figures and budget allocations?

  ROLE: FINANCE

    Chunk 1:
      collection   : finance
      source       : quarterly_financial_report.docx
      section      : Quarterly Comparison - Revenue Trend
      page         : 0
      score        : 0.5535
      access_roles : ['finance', 'c_level']
      words        : 89
      text preview : [Quarterly Comparison - Revenue Trend]
| Quarter            | Revenue (₹ Crore)   | QoQ Growth   | YoY Growth         | Cumulative   |
|--------------------|---------------------|--------------|--------------------|--------------|
| Q1 2024            | 183.0               | —            | +22.0%             | 183.0        |
| Q2 2024            | 192.0               | +4.9%        | +24.7%             | 375.0        |
| Q3 2024            | 203.0               | +5.7%        | +28.5%             | 578.0        |
| Q4 2024            | 205.0               |